# Random Forest Genre Classifier for FMA Small or Medium

This notebook builds a random forest genre classifier for either `fma_small` or `fma_medium` using the precomputed tabular audio features in `fma_metadata/features.csv`.

Pipeline:

1. Resolve the project paths and load FMA metadata plus tabular features.
2. Keep only the labelled tracks from the selected subset.
3. Preserve the official FMA training, validation, and test split.
4. Optionally cap each split per genre to keep experiments manageable and balanced.
5. Train a random forest over one or more `n_estimators` settings.
6. Save the checkpoint with the best validation accuracy.
7. Evaluate the best model on the held-out test split.


## 1. Setup

If the imports below fail in your Jupyter kernel, run this once in a notebook cell, restart the kernel, then continue:

```python
%pip install pandas numpy scikit-learn
```


In [15]:
from __future__ import annotations

import pickle
import random
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Resolve the dataset paths relative to either the current working directory or its parent.
PROJECT_CANDIDATES = [Path.cwd(), Path.cwd().parent]

for candidate in PROJECT_CANDIDATES:
    metadata_dir = candidate / "fma_metadata"
    tracks_path = metadata_dir / "tracks.csv"
    features_path = metadata_dir / "features.csv"
    if tracks_path.exists() and features_path.exists():
        PROJECT_DIR = candidate.resolve()
        TRACKS_PATH = tracks_path.resolve()
        FEATURES_PATH = features_path.resolve()
        break
else:
    raise FileNotFoundError(
        "Could not find fma_metadata/tracks.csv and fma_metadata/features.csv from the current directory or its parent."
    )

PROJECT_DIR


PosixPath('/home/lilyd/acml-project')

## 2. Load FMA Small Metadata and Features


In [16]:
def flatten_columns(columns: pd.Index) -> list[str]:
    return ["__".join(str(part) for part in column) for column in columns]


DATA_SUBSET = "medium"

if DATA_SUBSET not in {"small", "medium"}:
    raise ValueError("DATA_SUBSET must be either 'small' or 'medium'.")


def load_fma_features(subset: str) -> pd.DataFrame:
    # tracks.csv has a two-row header and features.csv has a three-row header.
    tracks = pd.read_csv(TRACKS_PATH, index_col=0, header=[0, 1])
    features = pd.read_csv(FEATURES_PATH, index_col=0, header=[0, 1, 2])
    features.columns = flatten_columns(features.columns)

    frame = pd.DataFrame({
        "track_id": tracks.index.astype(int),
        "subset": tracks[("set", "subset")],
        "split": tracks[("set", "split")],
        "genre": tracks[("track", "genre_top")],
    }).set_index("track_id")

    # Keep only the labelled rows from the selected official subset.
    frame = frame.join(features, how="inner")
    frame = frame[(frame["subset"] == subset) & frame["genre"].notna()].copy()
    return frame


metadata = load_fma_features(DATA_SUBSET)
metadata.head()


,subset,split,genre,chroma_cens__kurtosis__01,chroma_cens__kurtosis__02,chroma_cens__kurtosis__03,chroma_cens__kurtosis__04,chroma_cens__kurtosis__05,chroma_cens__kurtosis__06,chroma_cens__kurtosis__07,...,tonnetz__std__04,tonnetz__std__05,tonnetz__std__06,zcr__kurtosis__01,zcr__max__01,zcr__mean__01,zcr__median__01,zcr__min__01,zcr__skew__01,zcr__std__01
track_id,,,,,,,,,,,,,,,,,,,,,
3,medium,training,Hip-Hop,1.888963,0.760539,0.345297,2.295201,1.654031,0.067592,1.366848,...,0.063831,0.014212,0.017740,2.824694,0.466309,0.084578,0.063965,0.000000,1.716724,0.069330
134,medium,training,Hip-Hop,0.918445,0.674147,0.577818,1.281117,0.933746,0.078177,1.199204,...,0.058766,0.016322,0.015819,4.731087,0.419434,0.064370,0.050781,0.000000,1.806106,0.054623
136,medium,training,Rock,0.915001,-0.643476,-0.460507,-0.530701,-0.364460,-0.226860,-0.060377,...,0.076808,0.017915,0.016706,0.558770,0.147461,0.036686,0.034180,0.003418,0.805020,0.016905
139,medium,training,Folk,-0.020869,0.432330,0.331278,0.829845,2.625593,2.005660,0.907704,...,0.090518,0.017428,0.021490,1.157352,0.261230,0.070760,0.066895,0.000977,0.769163,0.030017
181,medium,test,Rock,-0.200982,-0.299499,0.848179,0.145290,0.364666,-0.004230,-0.270724,...,0.099341,0.021020,0.016658,0.260331,0.250977,0.086977,0.079102,0.000000,0.788906,0.037728


In [17]:
print(f"Usable FMA {DATA_SUBSET} rows: {len(metadata):,}")
print(metadata["split"].value_counts())
print(metadata["genre"].value_counts().sort_index())
print(f"Feature columns: {metadata.shape[1] - 3}")


Usable FMA medium rows: 17,000
split
training      13522
test           1773
validation     1705
Name: count, dtype: int64
genre
Blues                    74
Classical               619
Country                 178
Easy Listening           21
Electronic             5314
Experimental           1251
Folk                    519
Hip-Hop                1201
Instrumental            350
International            18
Jazz                    384
Old-Time / Historic     510
Pop                     186
Rock                   6103
Soul-RnB                154
Spoken                  118
Name: count, dtype: int64
Feature columns: 518


## 3. Pick a Manageable Balanced Split

Set `DATA_SUBSET` to either `"small"` or `"medium"`. The defaults below use the full official split for whichever subset you select. Set the caps to smaller integers when you want quicker experiments.


In [18]:
# Switch between the two official labelled subsets here.
DATA_SUBSET

# These caps keep classes balanced while letting you shrink the experiment when needed.
MAX_TRAIN_PER_GENRE = None
MAX_EVAL_PER_GENRE = None


def cap_per_genre(frame: pd.DataFrame, max_per_genre: int | None) -> pd.DataFrame:
    if frame.empty:
        return frame.reset_index(drop=False)
    if max_per_genre is None:
        return frame.sample(frac=1, random_state=SEED).reset_index(drop=False)

    sampled_groups = [
        group.sample(min(len(group), max_per_genre), random_state=SEED)
        for _, group in frame.groupby("genre", sort=False)
    ]
    return (
        pd.concat(sampled_groups)
        .sample(frac=1, random_state=SEED)
        .reset_index(drop=False)
    )


# Preserve the official FMA split instead of randomly mixing train, validation, and test tracks.
train_df = cap_per_genre(metadata[metadata["split"] == "training"], MAX_TRAIN_PER_GENRE)
val_df = cap_per_genre(metadata[metadata["split"] == "validation"], MAX_EVAL_PER_GENRE)
test_df = cap_per_genre(metadata[metadata["split"] == "test"], MAX_EVAL_PER_GENRE)

# Convert text genres to integer class ids for the classifier and evaluation.
genres = sorted(train_df["genre"].unique())
genre_to_idx = {genre: idx for idx, genre in enumerate(genres)}
idx_to_genre = {idx: genre for genre, idx in genre_to_idx.items()}

for frame in (train_df, val_df, test_df):
    frame["label"] = frame["genre"].map(genre_to_idx).astype(int)

print(f"Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")
print(genre_to_idx)


Train: 13,522 | Val: 1,705 | Test: 1,773
{'Blues': 0, 'Classical': 1, 'Country': 2, 'Easy Listening': 3, 'Electronic': 4, 'Experimental': 5, 'Folk': 6, 'Hip-Hop': 7, 'Instrumental': 8, 'International': 9, 'Jazz': 10, 'Old-Time / Historic': 11, 'Pop': 12, 'Rock': 13, 'Soul-RnB': 14, 'Spoken': 15}


## 4. Build Tabular Matrices


In [19]:
def split_features_and_labels(frame: pd.DataFrame) -> tuple[np.ndarray, np.ndarray, list[str]]:
    feature_columns = [
        column
        for column in frame.columns
        if column not in {"track_id", "subset", "split", "genre", "label"}
    ]
    x = frame[feature_columns].to_numpy(dtype=np.float32)
    y = frame["label"].to_numpy(dtype=np.int64)
    return x, y, feature_columns


X_train, y_train, feature_columns = split_features_and_labels(train_df)
X_val, y_val, _ = split_features_and_labels(val_df)
X_test, y_test, _ = split_features_and_labels(test_df)

X_train.shape, X_val.shape, X_test.shape


((13522, 518), (1705, 518), (1773, 518))

## 5. Train the Random Forest

The cell below tries multiple forest sizes and keeps the model with the best validation accuracy.

`ccp_alpha` is the direct post-pruning control. `max_depth`, `min_samples_split`, `min_samples_leaf`, and `max_leaf_nodes` also regularize tree growth and are usually the first knobs to tune.


In [20]:
# Tune these settings if you want to explore different random forest configurations.
N_ESTIMATOR_OPTIONS = [250]
MAX_DEPTH = 10
MAX_FEATURES = None
MIN_SAMPLES_SPLIT = 5
MIN_SAMPLES_LEAF = 1
MAX_LEAF_NODES = None
CCP_ALPHA = 0.0

best_val_acc = -1.0
best_model = None
best_n_estimators = None
history = []
best_model_path = PROJECT_DIR / "code" / "best_random_forest.pkl"

for n_estimators in N_ESTIMATOR_OPTIONS:
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=MAX_DEPTH,
        max_features=MAX_FEATURES,
        min_samples_split=MIN_SAMPLES_SPLIT,
        min_samples_leaf=MIN_SAMPLES_LEAF,
        max_leaf_nodes=MAX_LEAF_NODES,
        ccp_alpha=CCP_ALPHA,
        n_jobs=-1,
        random_state=SEED,
    )
    model.fit(X_train, y_train)

    train_predictions = model.predict(X_train)
    val_predictions = model.predict(X_val)
    train_acc = accuracy_score(y_train, train_predictions)
    val_acc = accuracy_score(y_val, val_predictions)

    history.append({
        "n_estimators": n_estimators,
        "train_acc": train_acc,
        "val_acc": val_acc,
    })

    print(
        f"n_estimators {n_estimators:4d} | "
        f"train acc {train_acc:.3f} | "
        f"val acc {val_acc:.3f}"
    )

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_model = model
        best_n_estimators = n_estimators
        with best_model_path.open("wb") as handle:
            pickle.dump({
                "model": best_model,
                "genre_to_idx": genre_to_idx,
                "idx_to_genre": idx_to_genre,
                "feature_columns": feature_columns,
                "config": {
                    "subset": DATA_SUBSET,
                    "max_train_per_genre": MAX_TRAIN_PER_GENRE,
                    "max_eval_per_genre": MAX_EVAL_PER_GENRE,
                    "n_estimators": best_n_estimators,
                    "max_depth": MAX_DEPTH,
                    "max_features": MAX_FEATURES,
                    "min_samples_split": MIN_SAMPLES_SPLIT,
                    "min_samples_leaf": MIN_SAMPLES_LEAF,
                    "max_leaf_nodes": MAX_LEAF_NODES,
                    "ccp_alpha": CCP_ALPHA,
                    "seed": SEED,
                },
                "history": history,
            }, handle)

pd.DataFrame(history)


n_estimators  250 | train acc 0.852 | val acc 0.701


,n_estimators,train_acc,val_acc
0,250,0.851649,0.70088


## 5b. Hyperparameter Tuning

Sweep a grid of values for each random forest knob and pick the configuration with the best validation accuracy. The grids below are small on purpose so the sweep stays tractable; widen them when you have more time. The best model from the sweep replaces the checkpoint saved above only if it beats the current best validation accuracy.

In [ ]:
from itertools import product

# Edit these grids to control the search space. Each list is the set of values to try for that hyperparameter.
PARAM_GRID = {
    "n_estimators": [100, 250, 500],
    "max_depth": [10, 20, None],
    "max_features": ["sqrt", "log2", None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_leaf_nodes": [None, 256],
    "ccp_alpha": [0.0, 1e-4],
}

param_names = list(PARAM_GRID.keys())
combinations = list(product(*(PARAM_GRID[name] for name in param_names)))
print(f"Trying {len(combinations)} configurations...")

tuning_history = []
tuned_best_val_acc = -1.0
tuned_best_model = None
tuned_best_params = None

for idx, values in enumerate(combinations, start=1):
    params = dict(zip(param_names, values))
    model = RandomForestClassifier(
        **params,
        n_jobs=-1,
        random_state=SEED,
    )
    model.fit(X_train, y_train)

    train_acc = accuracy_score(y_train, model.predict(X_train))
    val_acc = accuracy_score(y_val, model.predict(X_val))

    tuning_history.append({**params, "train_acc": train_acc, "val_acc": val_acc})

    print(
        f"[{idx:3d}/{len(combinations)}] "
        + " ".join(f"{name}={params[name]}" for name in param_names)
        + f" | train {train_acc:.3f} | val {val_acc:.3f}"
    )

    if val_acc > tuned_best_val_acc:
        tuned_best_val_acc = val_acc
        tuned_best_model = model
        tuned_best_params = params

tuning_results = (
    pd.DataFrame(tuning_history)
    .sort_values("val_acc", ascending=False)
    .reset_index(drop=True)
)

print("\nBest configuration from sweep:")
for name, value in tuned_best_params.items():
    print(f"  {name}: {value}")
print(f"  validation accuracy: {tuned_best_val_acc:.3f}")

# Promote the tuned model to the saved checkpoint only if it beats the previous best.
if tuned_best_val_acc > best_val_acc:
    print(f"\nTuned model improved val acc {best_val_acc:.3f} -> {tuned_best_val_acc:.3f}. Updating checkpoint.")
    best_val_acc = tuned_best_val_acc
    best_model = tuned_best_model
    best_n_estimators = tuned_best_params["n_estimators"]
    with best_model_path.open("wb") as handle:
        pickle.dump({
            "model": best_model,
            "genre_to_idx": genre_to_idx,
            "idx_to_genre": idx_to_genre,
            "feature_columns": feature_columns,
            "config": {
                "subset": DATA_SUBSET,
                "max_train_per_genre": MAX_TRAIN_PER_GENRE,
                "max_eval_per_genre": MAX_EVAL_PER_GENRE,
                **tuned_best_params,
                "seed": SEED,
            },
            "history": history,
            "tuning_history": tuning_history,
        }, handle)
else:
    print(f"\nNo tuned model beat the existing best ({best_val_acc:.3f}); checkpoint unchanged.")

tuning_results.head(10)

## 6. Test Evaluation


In [ ]:
# Reload the best validation checkpoint before measuring final test performance.
with best_model_path.open("rb") as handle:
    checkpoint = pickle.load(handle)

best_model = checkpoint["model"]
test_predictions = best_model.predict(X_test)
test_acc = accuracy_score(y_test, test_predictions)

print(f"Best validation accuracy: {best_val_acc:.3f} at n_estimators={best_n_estimators}")
print(f"Test accuracy: {test_acc:.3f}")


## 7. Inspect Predictions


In [ ]:
pd.DataFrame({
    "true": [idx_to_genre[int(label)] for label in y_test[:16]],
    "predicted": [idx_to_genre[int(prediction)] for prediction in test_predictions[:16]],
})


In [ ]:
print("Classification report:")
print(classification_report(y_test, test_predictions, target_names=genres, digits=3))

confusion = confusion_matrix(y_test, test_predictions)
confusion_df = pd.DataFrame(confusion, index=genres, columns=genres)
confusion_df


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(12, 10))
image = ax.imshow(confusion, cmap="Blues")
fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)

ax.set_title("Confusion Matrix")
ax.set_xlabel("Predicted label")
ax.set_ylabel("True label")
ax.set_xticks(range(len(genres)))
ax.set_yticks(range(len(genres)))
ax.set_xticklabels(genres, rotation=90)
ax.set_yticklabels(genres)

for i in range(confusion.shape[0]):
    for j in range(confusion.shape[1]):
        ax.text(j, i, confusion[i, j], ha="center", va="center", color="black", fontsize=8)

plt.tight_layout()
plt.show()
